In [ ]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

import xgboost as xgb
import pandas as pd
import numpy as np
import cudf
import cuml
import json
import tensorflow as tf
import seaborn as sns


from tensorflow import keras
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, balanced_accuracy_score
from cuml.ensemble import RandomForestClassifier as cuRF
from pathlib import Path

In [ ]:
for index in range (0, 41): 
    ARCHIVO_ORIGEN = "muestras_reales.csv"
    ARCHIVO_SALIDA = "dataset_sintetico.csv"
    COLUMNA_CLASE  = "Tipo"

    # Los 18 canales espectrales
    CANALES = ["A_410", "B_435", "C_460", "D_485", "E_510", "F_535",
            "G_560", "H_585", "R_610", "I_645", "S_680", "J_705",
            "T_730", "U_760", "V_810", "W_860", "K_900", "L_940"]

    N_SINTETICAS   = 953        #Esto nos genera unas 40036 muestras
    SIGMA          = 0.01
    RUIDO_RELATIVO = False      # Lo ponemos a false para que no mantengga la estructura estadística
    SEMILLA        = 42

    rng = np.random.default_rng(SEMILLA)

    df = pd.read_csv(Path(ARCHIVO_ORIGEN), encoding="latin-1", sep=";", decimal=",")

    prueba_dato_real= df.loc[index].to_list()

    df.drop(index)

    X = df[CANALES].to_numpy(dtype=np.float64)
    y = df[COLUMNA_CLASE].to_numpy()


    filas_sinteticas, etiquetas_sinteticas = [], []
    for firma, etiqueta in zip(X, y):
        sigma_vec = SIGMA * np.abs(firma) if RUIDO_RELATIVO else SIGMA
        ruido = rng.normal(loc=0.0, scale=sigma_vec, size=(N_SINTETICAS, X.shape[1]))
        filas_sinteticas.append(firma + ruido)
        etiquetas_sinteticas.extend([etiqueta] * N_SINTETICAS)

    X_sint = np.vstack(filas_sinteticas)

    X_final = np.vstack([X, X_sint])
    y_final = np.concatenate([y, etiquetas_sinteticas])

    df_final = pd.DataFrame(X_final, columns=CANALES)
    df_final[COLUMNA_CLASE] = y_final
    df_final = df_final.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

    df_final.to_csv(Path(ARCHIVO_SALIDA), index=False, encoding="utf-8-sig")

    ### Parte de carga de datos nuevos


    df = pd.read_csv("dataset_sintetico.csv")

    COLS_ELIMINAR = ['Tipo', 'L', 'B', 'A']
    X = df.drop(COLS_ELIMINAR, axis=1)
    y = df['Tipo']

    class_counts = y.value_counts()
    clases_validas = class_counts[class_counts >= 10].index
    mask = y.isin(clases_validas)

    X_filt = X[mask].copy()
    y_filt = y[mask].copy()

    clases_eliminadas = class_counts[class_counts < 10].index.tolist()

    nan_cols = X_filt.columns[X_filt.isna().any()].tolist()


    le = LabelEncoder()
    y_enc = le.fit_transform(y_filt).astype(np.int32)


    X_train, X_test, y_train, y_test = train_test_split(
        X_filt, y_enc,
        test_size=0.2,
        random_state=42,
        stratify=y_enc
    )

    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp  = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp).astype(np.float32)
    X_test_scaled  = scaler.transform(X_test_imp).astype(np.float32)


    ### MLP entrenamiento y comparación con resultado real

    mlp = MLPClassifier(
        hidden_layer_sizes=(100,50,50),
        activation='tanh',
        solver='sgd',
        max_iter=1000,
        alpha = 0.001,
        random_state=42,
        learning_rate='adaptive',
    )

    from sklearn.utils.class_weight import compute_sample_weight
    sw = compute_sample_weight('balanced', y_train)
    mlp.fit(X_train_scaled, y_train)
    predicciones = mlp.predict(X_test_scaled)

    precision = accuracy_score(y_test, predicciones)
    print(f"La precisión de la red es de: {precision * 100}%")
    print(f"El dato real es: {prueba_dato_real}")
    precision_real = mlp.predict(prueba_dato_real.reshape(1, -1))

    ### XGBOOSt entrenamiento y comparativa con resultado real

    model = xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric='mlogloss',
        tree_method='hist',
        device='cuda',
        random_state=42
    )

    param_grid = {
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'n_estimators': [100, 200],
        'subsample': [0.8, 1.0]
    }

    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv = 5, scoring='accuracy')

    grid_search.fit(X_train_scaled, y_train)

    print(f"Mejor configuración: {grid_search.best_params_}")
    print(f"Mejor score de validación: {grid_search.best_score_:.4f}")
    print(f"El dato real es: {prueba_dato_real}")
    precision_real = grid_search.best_estimator_.predict(prueba_dato_real.reshape(1, -1))

    ### Modelo híbrido: TF y XGboost

    best_model = grid_search.best_estimator_
    soft_labels = best_model.predict_proba(X_train_scaled)  # shape (n_samples, 7)

    n_features = X_train_scaled.shape[1]
    n_clases = soft_labels.shape[1]

    #Definir red Keras
    model_keras = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_features,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(n_clases, activation='softmax')
    ])

    model_keras.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    #Entrenar imitando al XGBoost
    model_keras.fit(
        X_train_scaled, soft_labels,
        epochs=50,
        batch_size=32,
        validation_split=0.1,
        verbose=1
    )

    #Evaluar vs el XGBoost original
    y_pred_keras = np.argmax(model_keras.predict(X_test_scaled), axis=1)
    y_pred_xgb   = best_model.predict(X_test_scaled)

    print(f"Accuracy Keras:   {accuracy_score(y_test, y_pred_keras):.4f}")
    print(f"Accuracy XGBoost: {accuracy_score(y_test, y_pred_xgb):.4f}")
    print(f"El dato real es: {prueba_dato_real}")
    precision_real = model_keras.predict(prueba_dato_real.reshape(1, -1))

    #Guardar en formato Keras
    model_keras.save('mejor_modelo.keras')

    with open('clases.json', 'w', encoding='utf-8') as f:
        json.dump(le.classes_.tolist(), f, ensure_ascii=False)
    print(f"Clases guardadas ({len(le.classes_)}): {le.classes_.tolist()}")